# Mengukur Jarak Tipe Data Campuran (Studi Kasus: Travel Insurance)

Dalam dunia nyata, dataset seringkali memiliki tipe data yang beragam (Mixed Data Types). Kita tidak bisa menggunakan Euclidean atau Manhattan saja jika terdapat data kategorikal. Salah satu metode yang paling efektif untuk menangani hal ini adalah **Gower Distance**.

Pada bagian ini, kita akan menganalisis dataset **Travel Insurance Prediction** dari Kaggle untuk mengukur tingkat kemiripan antar pelanggan berdasarkan profil mereka.

## 1. Data Understanding
Dataset ini berisi informasi pelanggan perusahaan asuransi perjalanan. Tujuan utamanya adalah memprediksi apakah seorang pelanggan akan membeli asuransi atau tidak. Namun, di sini kita fokus pada bagaimana mengukur jarak (kemiripan) antar pelanggan tersebut.

### **Deskripsi Atribut yang Digunakan:**
* **Age** (Numerik): Umur pelanggan (rentang 25 - 35 tahun).
* **Employment Type** (Kategorikal): Sektor pekerjaan (Government / Private).
* **GraduateOrNot** (Biner): Status lulus kuliah (Yes / No).
* **AnnualIncome** (Numerik): Pendapatan tahunan (rentang 300.000 - 1.800.000).
* **EverTravelledAbroad** (Biner): Pernah ke luar negeri (Yes / No).

In [6]:
import pandas as pd
import numpy as np

# Load dataset yang sudah diupload
df = pd.read_csv('TravelInsurancePrediction.csv')

# Pilih 5 fitur utama dan tampilkan 3 data pertama sebagai sampel
df_subset = df[['Age', 'Employment Type', 'GraduateOrNot', 'AnnualIncome', 'EverTravelledAbroad']].head(3).copy()
display(df_subset)

,Age,Employment Type,GraduateOrNot,AnnualIncome,EverTravelledAbroad
0,31,Government Sector,Yes,400000,No
1,31,Private Sector/Self Employed,Yes,1250000,No
2,34,Private Sector/Self Employed,Yes,500000,No


## 2. Transformasi Data

Sebelum menghitung jarak Gower, kita perlu melakukan transformasi:
1.  **Numerik:** Menggunakan normalisasi agar selisihnya berada di rentang 0-1.
2.  **Kategorikal/Biner:** Menggunakan perbandingan kecocokan (Match = 0, Mismatch = 1).

### **Normalisasi Min-Max untuk Numerik**
Rumus yang digunakan:
$$x' = \frac{|x_i - x_j|}{Range}$$
Di mana *Range* adalah selisih nilai maksimum dan minimum pada atribut tersebut.

* **Range Age:** $35 - 25 = 10$
* **Range AnnualIncome:** $1.800.000 - 300.000 = 1.500.000$

## 3. Perhitungan Jarak Gower (Manual)

Untuk memahami bagaimana Gower Distance bekerja, kita akan menghitung jarak antara **Data 1** (Index 0) dan **Data 2** (Index 1) secara manual.

**Profil Data:**
* **Data 1 ($x_1$):** Age=31, Employment=Gov, Graduate=Yes, Income=400k, Abroad=No
* **Data 2 ($x_2$):** Age=31, Employment=Private, Graduate=Yes, Income=1.25M, Abroad=No

### **Langkah 1: Menghitung Selisih per Atribut ($s_{ij}$)**

1.  **Atribut Age (Numerik):**
    $$s_{12}^{(Age)} = \frac{|31 - 31|}{10} = \frac{0}{10} = 0$$

2.  **Atribut Employment Type (Kategorikal):**
    Karena 'Government Sector' $\neq$ 'Private Sector/Self Employed', maka:
    $$s_{12}^{(Employment)} = 1$$

3.  **Atribut GraduateOrNot (Biner):**
    Karena 'Yes' = 'Yes', maka:
    $$s_{12}^{(Graduate)} = 0$$

4.  **Atribut AnnualIncome (Numerik):**
    $$s_{12}^{(Income)} = \frac{|400.000 - 1.250.000|}{1.500.000} = \frac{850.000}{1.500.000} \approx 0.566$$

5.  **Atribut EverTravelledAbroad (Biner):**
    Karena 'No' = 'No', maka:
    $$s_{12}^{(Abroad)} = 0$$

### **Langkah 2: Menghitung Rata-rata Jarak Gower**

Rumus umum Gower Distance:
$$d(i,j) = \frac{\sum_{k=1}^{n} w_k s_{ij}^{(k)}}{\sum_{k=1}^{n} w_k}$$

Dengan bobot ($w_k$) masing-masing atribut adalah 1, maka:
$$d(1,2) = \frac{0 + 1 + 0 + 0.566 + 0}{5}$$
$$d(1,2) = \frac{1.566}{5} = \mathbf{0.3132}$$

**Kesimpulan Manual:** Tingkat ketidakmiripan antara Pelanggan 1 dan Pelanggan 2 adalah sebesar **0.3132**.

In [7]:
import pandas as pd
import numpy as np

# 1. Pastikan data sudah dibaca
df = pd.read_csv('TravelInsurancePrediction.csv')
df_subset = df[['Age', 'Employment Type', 'GraduateOrNot', 'AnnualIncome', 'EverTravelledAbroad']].head(3).copy()

# 2. Definisikan fungsi Gower
def gower_distance(row1, row2, types, ranges):
    dist = 0
    for i, col in enumerate(row1.index):
        if types[i] == 'num':
            # Rumus: |x1 - x2| / Range
            dist += abs(row1[col] - row2[col]) / ranges[col]
        else:
            # Rumus: 1 jika beda, 0 jika sama
            dist += 1 if row1[col] != row2[col] else 0
    return dist / len(row1)

# 3. Setting Range dan Tipe Data (Sesuai dataset Travel Insurance)
ranges = {'Age': 10, 'AnnualIncome': 1500000}
types = ['num', 'cat', 'cat', 'num', 'cat']

# 4. Hitung matriks jarak
n = len(df_subset)
matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        matrix[i,j] = gower_distance(df_subset.iloc[i], df_subset.iloc[j], types, ranges)

# 5. Tampilkan hasil
df_gower = pd.DataFrame(matrix, 
                        columns=['Pelanggan 1', 'Pelanggan 2', 'Pelanggan 3'], 
                        index=['Pelanggan 1', 'Pelanggan 2', 'Pelanggan 3'])

print("Matriks Jarak Gower (3 Sampel Pelanggan):")
display(df_gower)

Matriks Jarak Gower (3 Sampel Pelanggan):


,Pelanggan 1,Pelanggan 2,Pelanggan 3
Pelanggan 1,0.000000,0.313333,0.273333
Pelanggan 2,0.313333,0.000000,0.160000
Pelanggan 3,0.273333,0.160000,0.000000


## 4. Implementasi Menggunakan Orange Data Mining

Di Orange, kita bisa mendapatkan hasil yang sama dengan lebih cepat menggunakan widget **Distances**.

**Langkah-langkah:**
1.  Gunakan widget **File** untuk memuat dataset `TravelInsurancePrediction.csv`.
2.  Pastikan di **Edit Domain**, kolom Age dan Income bertipe *Numeric*, sedangkan yang lain *Categorical*.
3.  Tarik ke widget **Distances** dan pilih metrik **Gower**.
4.  Tampilkan di widget **Distance Matrix**.

![Workflow Orange](workflow_campuran.png)
![Matriks Gower Orange](gower_matrix.png)